# W3 Results Evaluation

Evaluate baselines against brute-force ground truth for W3 (multi-signal semantic retrieval, no predicates).

**Recall**: For each query, count how many baseline results have score ≤ worst GT score. recall = hits / K.

In [1]:
import json
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(".")
GT_FILE = Path("w3_queries_100_gt.json")

def load_results(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def gt_answer_to_scores(answer: list) -> list:
    return [(row["id"], row["score"]) for row in answer]

def baseline_answer_to_scores(answer: list) -> list:
    return [(row[0], row[-1]) for row in answer]

def query_results_to_map(data: dict, is_gt: bool = False) -> dict:
    fn = gt_answer_to_scores if is_gt else baseline_answer_to_scores
    return {r["query_id"]: fn(r["answer"]) for r in data["results"]}

assert GT_FILE.exists(), f"Ground truth not found: {GT_FILE}"
gt_data = load_results(GT_FILE)
gt_by_qid = query_results_to_map(gt_data, is_gt=True)

BASELINE_FILES = sorted(
    f.name for f in RESULTS_DIR.glob("*.json")
    if f.name != GT_FILE.name
)

print(f"Ground truth: {GT_FILE.name}, queries = {len(gt_by_qid)}")
print(f"Baselines ({len(BASELINE_FILES)}):")
for f in BASELINE_FILES:
    print(f"  {f}")

Ground truth: w3_queries_100_gt.json, queries = 94
Baselines (5):
  20260414_062856.json
  20260414_062911.json
  20260414_062936.json
  20260414_062956.json
  20260414_063017.json


In [2]:
def eval_one_baseline(baseline_by_qid: dict, gt_by_qid: dict):
    per_query = []
    total_hits = 0
    total_denom = 0
    for qid, answers in baseline_by_qid.items():
        if qid not in gt_by_qid:
            continue
        gt_answers = gt_by_qid[qid]
        n = len(gt_answers)
        if n == 0:
            continue
        an = max(score for _, score in gt_answers)
        m = sum(1 for _, score in answers if score <= an + 1e-6)
        recall = m / n
        per_query.append({"query_id": qid, "K": n, "hits": m, "recall": recall})
        total_hits += m
        total_denom += n
    mean_recall = total_hits / total_denom if total_denom else 0.0
    return per_query, mean_recall

In [3]:
summary = []
for f in BASELINE_FILES:
    data = load_results(RESULTS_DIR / f)
    by_qid = query_results_to_map(data, is_gt=False)
    per_query, mean_recall = eval_one_baseline(by_qid, gt_by_qid)
    name = Path(f).stem
    n_queries = len(per_query)
    total_sec = data.get("total_elapsed_sec", None)
    qps = n_queries / total_sec if total_sec and total_sec > 0 else 0.0
    summary.append({
        "baseline": name,
        "mean_recall": mean_recall,
        "n_queries": n_queries,
        "qps": qps,
    })
    print(f"{name}: mean_recall = {mean_recall:.4f}, qps = {qps:.2f}")

20260414_062856: mean_recall = 0.9856, qps = 36.25
20260414_062911: mean_recall = 0.9856, qps = 35.48
20260414_062936: mean_recall = 0.9856, qps = 36.90
20260414_062956: mean_recall = 0.9856, qps = 36.28
20260414_063017: mean_recall = 0.9856, qps = 36.45


In [4]:
import pandas as pd

df = pd.DataFrame(summary).sort_values("mean_recall", ascending=False)
print(df.to_string(index=False))

       baseline  mean_recall  n_queries       qps
20260414_062856     0.985638         94 36.245614
20260414_062911     0.985638         94 35.484352
20260414_062936     0.985638         94 36.904334
20260414_062956     0.985638         94 36.280104
20260414_063017     0.985638         94 36.450597
